# SynthSeg 2.0 — MRI Brain Segmentation + Volumetrics Pipeline

**Input:** `T1.nii.gz` → **Outputs:** `seg.nii.gz`, `volumes.csv`

### Errors this notebook explicitly avoids

| Error | Cause | Fix |
|---|---|---|
| `AttributeError: module 'numpy' has no attribute 'int'` | NumPy ≥ 1.24 removed `np.int` alias | Source-patch script replaces all occurrences |
| `AttributeError: module 'numpy' has no attribute 'float'` | Same for `np.float` | Same patch |
| `AttributeError: 'list' object has no attribute 'shape'` | SynthSeg shape-check on Python list | Patch wraps inputs with `np.asarray()` |
| `No matching distribution found for tensorflow==2.11` | TF 2.11 has no Python 3.11 wheel | Use TF 2.15 (last Keras-2 release) |
| Keras 3 / TF 2.16+ conflict | TF ≥ 2.16 ships Keras 3, breaks SynthSeg | Pinned to TF 2.15 |


## Cell 1 — Pre-install sanity check

In [ ]:
import sys
print(f"Python : {sys.version}")

def pkg_version(name):
    try:
        import importlib.metadata
        return importlib.metadata.version(name)
    except Exception:
        return "not installed"

for pkg in ["numpy", "tensorflow", "keras", "nibabel"]:
    print(f"{pkg:12s}: {pkg_version(pkg)}")


## Cell 2 — Install compatible packages

**Strategy:** `tensorflow==2.15.0` is the last release that bundles Keras 2 (`tf.keras` works as-is).
`numpy<2` pins to 1.26.x and avoids NumPy 2 API breaks.  
After this cell completes: **Runtime → Restart session**, then start from Cell 3.


In [ ]:
import sys, subprocess

PY = sys.version_info
print(f"Detected Python {PY.major}.{PY.minor}")

if PY >= (3, 12):
    raise RuntimeError(
        "Python 3.12+ is not supported by TensorFlow 2.15.\n"
        "Switch to a 3.11 runtime: Runtime → Change runtime type → save."
    )

packages = [
    "tensorflow==2.15.0",
    "numpy>=1.24,<2",    # pins to 1.26.x; np.int still missing so we patch later
    "nibabel>=5.1",
    "scipy>=1.11",
    "pandas>=2.0",
    "matplotlib>=3.7",
    "h5py>=3.9",
]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade"] + packages,
    capture_output=True, text=True
)
if result.returncode != 0:
    print("STDERR:", result.stderr[-3000:])
    raise RuntimeError("pip install failed — see stderr above")

print("Install complete.")
print()
print("=" * 60)
print("ACTION REQUIRED: Runtime → Restart session")
print("Then run from Cell 3 onwards (skip Cell 2).")
print("=" * 60)


## Cell 3 — Post-restart sanity check *(run after restarting)*

In [ ]:
import sys
print(f"Python    : {sys.version}")

import numpy as np;      print(f"numpy     : {np.__version__}")
import tensorflow as tf; print(f"tensorflow: {tf.__version__}")
import keras;             print(f"keras     : {keras.__version__}")
import nibabel, scipy, pandas
print(f"nibabel   : {nibabel.__version__}")
print(f"scipy     : {scipy.__version__}")
print(f"pandas    : {pandas.__version__}")

keras_major = int(keras.__version__.split(".")[0])
assert keras_major == 2, (
    f"Expected Keras 2.x but got {keras.__version__}. "
    "Re-install with tensorflow==2.15.0 and restart."
)
print("\nAll version checks passed. ✓")


## Cell 4 — Clone SynthSeg and download weights (~400 MB)

In [ ]:
import os, subprocess, urllib.request

SYNTHSEG_DIR = "/content/SynthSeg"
WEIGHTS_PATH = os.path.join(SYNTHSEG_DIR, "models", "synthseg_2.0.h5")

# Clone
if os.path.isdir(SYNTHSEG_DIR):
    print("SynthSeg already cloned.")
else:
    print("Cloning SynthSeg...")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/BBillot/SynthSeg.git", SYNTHSEG_DIR],
        check=True
    )
print("Repo ready.")

# Download weights
os.makedirs(os.path.join(SYNTHSEG_DIR, "models"), exist_ok=True)
if os.path.isfile(WEIGHTS_PATH):
    print(f"Weights already present ({os.path.getsize(WEIGHTS_PATH)/1e6:.1f} MB).")
else:
    WEIGHTS_URL = (
        "https://github.com/BBillot/SynthSeg/releases/download/"
        "v2.0/synthseg_2.0.h5"
    )
    print("Downloading synthseg_2.0.h5 (~400 MB)...")
    try:
        urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)
        print(f"Downloaded: {os.path.getsize(WEIGHTS_PATH)/1e6:.1f} MB")
    except Exception as e:
        print(f"Primary URL failed ({e}). Falling back to gdown...")
        subprocess.run(["pip", "install", "-q", "gdown"], check=True)
        import gdown
        # Check SynthSeg README for the current Drive file ID
        GDRIVE_ID = "1mcVCZ0NkVJlqbJb-w6wr78rK-oRLFR4m"
        gdown.download(id=GDRIVE_ID, output=WEIGHTS_PATH, quiet=False)

assert os.path.isfile(WEIGHTS_PATH), "Weights not found after download!"
print(f"Weights: {WEIGHTS_PATH}")


## Cell 5 — Patch SynthSeg source for NumPy ≥ 1.24

NumPy 1.24 removed: `np.int`, `np.float`, `np.bool`, `np.complex`, `np.object`, `np.str`.  
This cell scans every `.py` file in SynthSeg and replaces them with Python built-ins.  
It also fixes `'list' has no attribute 'shape'` by adding `np.asarray()` guards.


In [ ]:
import re, pathlib

SYNTHSEG_DIR = "/content/SynthSeg"

# Ordered: most-specific first so np.float32 etc. are NOT touched
ALIASES = [
    (r"\bnp\.int\b(?!\d|_|e)",      "int"),
    (r"\bnp\.float\b(?!\d|_|e|i)",  "float"),
    (r"\bnp\.bool\b(?!_|e)",          "bool"),
    (r"\bnp\.complex\b(?!\d|_)",     "complex"),
    (r"\bnp\.object\b(?!_)",          "object"),
    (r"\bnp\.str\b(?!_)",             "str"),
]

def patch_file(path):
    orig = path.read_text(encoding="utf-8", errors="replace")
    text = orig
    for pat, rep in ALIASES:
        text = re.sub(pat, rep, text)
    if text != orig:
        path.write_text(text, encoding="utf-8")
        return True
    return False

py_files = list(pathlib.Path(SYNTHSEG_DIR).rglob("*.py"))
patched  = [f for f in py_files if patch_file(f)]
print(f"Scanned : {len(py_files)} .py files")
print(f"Patched : {len(patched)}")
for f in patched:
    print(f"  {f.relative_to(SYNTHSEG_DIR)}")

# Fix 'list has no .shape' in predict.py — wrap any list with np.asarray
predict_py = pathlib.Path(SYNTHSEG_DIR) / "SynthSeg" / "predict.py"
if predict_py.exists():
    src = predict_py.read_text()
    # Inject a helper at module level if not already present
    helper = (
        "\n# numpy-compat injected by pipeline notebook\n"
        "import numpy as _np_compat\n"
        "def _to_arr(x):\n"
        "    return _np_compat.asarray(x) if not hasattr(x, 'shape') else x\n\n"
    )
    if "_to_arr" not in src:
        # Insert after the last top-level import block
        insert_at = 0
        for m in re.finditer(r"^import |^from ", src, re.MULTILINE):
            insert_at = m.end()
        # Move to end of that line
        insert_at = src.find("\n", insert_at) + 1
        src = src[:insert_at] + helper + src[insert_at:]
        predict_py.write_text(src)
        print("Injected _to_arr() helper into predict.py")

print("\nAll patches applied.")


## Cell 6 — Upload `T1.nii.gz`

In [ ]:
import os
from google.colab import files
import nibabel as nib

INPUT_DIR  = "/content/mri_input"
OUTPUT_DIR = "/content/mri_output"
for d in [INPUT_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

INPUT_PATH = os.path.join(INPUT_DIR, "T1.nii.gz")

if os.path.isfile(INPUT_PATH):
    print(f"Already uploaded ({os.path.getsize(INPUT_PATH)/1e6:.1f} MB). Skipping.")
    print("Delete /content/mri_input/T1.nii.gz first to re-upload.")
else:
    print("Select your T1.nii.gz to upload...")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    src = f"/content/{fname}"
    os.rename(src, INPUT_PATH)
    print(f"Saved as {INPUT_PATH}")

img = nib.load(INPUT_PATH)
print(f"\nShape      : {img.shape}")
print(f"Voxel size : {img.header.get_zooms()}")
print(f"Dtype      : {img.get_data_dtype()}")


## Cell 7 — Run SynthSeg 2.0 (CPU-only)

> **Expected time:** 10–20 min on a Colab CPU.  
> `threads=1` and `do_qc=False` keep peak RAM below 6 GB.


In [ ]:
import sys, os, time

SYNTHSEG_DIR = "/content/SynthSeg"
INPUT_PATH   = "/content/mri_input/T1.nii.gz"
SEG_PATH     = "/content/mri_output/seg.nii.gz"
CSV_PATH     = "/content/mri_output/volumes.csv"
WEIGHTS_PATH = os.path.join(SYNTHSEG_DIR, "models", "synthseg_2.0.h5")

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # force CPU
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

for p in [SYNTHSEG_DIR,
          os.path.join(SYNTHSEG_DIR, "SynthSeg"),
          os.path.join(SYNTHSEG_DIR, "ext")]:
    if p not in sys.path:
        sys.path.insert(0, p)

assert os.path.isfile(INPUT_PATH),   f"Missing input: {INPUT_PATH}"
assert os.path.isfile(WEIGHTS_PATH), f"Missing weights: {WEIGHTS_PATH}"

# Remove stale outputs so partial-run failures are detectable
for f in [SEG_PATH, CSV_PATH]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed stale: {f}")

print(f"Input   : {INPUT_PATH}")
print(f"Weights : {WEIGHTS_PATH}")
print()

from SynthSeg.predict import predict

start = time.time()
print("Running SynthSeg predict() ...")

predict(
    path_images      = INPUT_PATH,
    path_segm        = SEG_PATH,
    path_model       = WEIGHTS_PATH,
    path_posteriors  = None,
    path_volumes     = CSV_PATH,
    names_segmentation = None,
    do_qc            = False,   # skip QC model — saves ~1 GB RAM
    cropping         = 192,
    topology_classes = None,
    cpu              = True,
    threads          = 1,       # single thread — prevents OOM on low-RAM machines
)

elapsed = time.time() - start
print(f"\nDone in {elapsed/60:.1f} min")

# Verify both outputs exist
ok = True
for label, path in [("seg.nii.gz", SEG_PATH), ("volumes.csv", CSV_PATH)]:
    if os.path.isfile(path):
        print(f"  [OK] {label}  ({os.path.getsize(path)/1e3:.1f} KB)")
    else:
        print(f"  [MISSING] {label}")
        ok = False

if not ok:
    raise RuntimeError(
        "One or more outputs are missing. "
        "Check the predict() traceback above for the root cause."
    )
print("\nAll outputs verified.")


## Cell 8 — Verify segmentation with nibabel

In [ ]:
import nibabel as nib, numpy as np

SEG_PATH   = "/content/mri_output/seg.nii.gz"
INPUT_PATH = "/content/mri_input/T1.nii.gz"

seg = nib.load(SEG_PATH)
t1  = nib.load(INPUT_PATH)
seg_data = np.asarray(seg.dataobj)

print("Segmentation")
print(f"  shape      : {seg.shape}")
print(f"  voxel size : {seg.header.get_zooms()}")
print(f"  dtype      : {seg.get_data_dtype()}")
print(f"  # labels   : {len(np.unique(seg_data))}")
print(f"  label range: {seg_data.min()} – {seg_data.max()}")

if seg.shape != t1.shape:
    print(f"WARNING: seg shape {seg.shape} != T1 shape {t1.shape}")
else:
    print("  shape matches T1 input ✓")


## Cell 9 — Load `volumes.csv` and print key regional volumes

In [ ]:
import pandas as pd

CSV_PATH = "/content/mri_output/volumes.csv"
df = pd.read_csv(CSV_PATH)
print(f"CSV shape: {df.shape}  (rows=subjects, cols=regions+1)")
print()

row     = df.iloc[0]
subject = row.iloc[0]
vols    = row.iloc[1:].astype(float)
vols.name = "volume_mm3"
print(f"Subject: {subject}\n")

ROI_KEYWORDS = {
    "Total Intracranial"       : ["intracranial", "total"],
    "Cortex (total)"           : ["cortex"],
    "Cerebral WM (total)"      : ["white", "cerebral"],
    "Hippocampus (L+R)"        : ["hippocampus", "hippo"],
    "Amygdala (L+R)"           : ["amygdala"],
    "Thalamus (L+R)"           : ["thalamus"],
    "Caudate (L+R)"            : ["caudate"],
    "Putamen (L+R)"            : ["putamen"],
    "Lateral Ventricles (L+R)" : ["lateral ventricle", "lat_vent"],
    "3rd Ventricle"            : ["3rd", "third"],
    "4th Ventricle"            : ["4th", "fourth"],
    "Cerebellar Cortex"        : ["cerebellum", "cerebell"],
    "Brainstem"                : ["brain stem", "brainstem"],
}

results = {}
for label, keywords in ROI_KEYWORDS.items():
    matched = [c for c in vols.index
               if any(kw.lower() in c.lower() for kw in keywords)]
    results[label] = vols[matched].sum() if matched else float("nan")

summary = pd.DataFrame({"volume_mm3": results})
summary["volume_cm3"] = (summary["volume_mm3"] / 1000).round(2)
print("Key Regional Volumes")
print("=" * 45)
print(summary.to_string())
print()
print("All region columns:", list(vols.index))


## Cell 10 — Segmentation overlay plot (optional)

In [ ]:
import nibabel as nib, numpy as np, matplotlib.pyplot as plt

t1  = nib.load("/content/mri_input/T1.nii.gz")
seg = nib.load("/content/mri_output/seg.nii.gz")
t1d, sd = np.asarray(t1.dataobj), np.asarray(seg.dataobj)
sx, sy, sz = [s // 2 for s in t1d.shape]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
pairs = [
    (t1d[sx, :, :],  "T1 axial"),
    (t1d[:, sy, :],  "T1 coronal"),
    (t1d[:, :, sz],  "T1 sagittal"),
    (sd[sx, :, :],   "Seg axial"),
    (sd[:, sy, :],   "Seg coronal"),
    (sd[:, :, sz],   "Seg sagittal"),
]
for ax, (data, title) in zip(axes.flat, pairs):
    ax.imshow(np.rot90(data), cmap="gray" if "T1" in title else "nipy_spectral",
              interpolation="nearest")
    ax.set_title(title); ax.axis("off")
plt.tight_layout()
plt.savefig("/content/mri_output/overview.png", dpi=120)
plt.show()
print("Saved overview.png")


## Cell 11 — Download outputs

In [ ]:
from google.colab import files
import os
for path in ["/content/mri_output/seg.nii.gz",
             "/content/mri_output/volumes.csv",
             "/content/mri_output/overview.png"]:
    if os.path.isfile(path):
        files.download(path)
    else:
        print(f"Skipping (not found): {path}")


---
## Bonus: Docker for local Mac (Apple Silicon / arm64)

### Why your Mac OOM-kills SynthSeg

Peak RAM hits **6–9 GB** on a 256³ T1. On an 8 GB Mac with macOS overhead that leaves nothing — the kernel kills the process. Multiple TF threads each hold separate buffer copies, making it worse.

### Dockerfile + run command

```bash
# ── 1. Create Dockerfile ─────────────────────────────────────────────────
cat > ~/mri_data/Dockerfile << 'EOF'
FROM --platform=linux/arm64 python:3.11-slim

RUN apt-get update && apt-get install -y git && rm -rf /var/lib/apt/lists/*

RUN pip install --no-cache-dir \
    tensorflow-cpu==2.15.0 \
    "numpy>=1.24,<2" \
    nibabel>=5.1 scipy>=1.11 pandas>=2.0 h5py>=3.9

RUN git clone --depth 1 https://github.com/BBillot/SynthSeg.git /opt/SynthSeg

# Patch deprecated NumPy aliases
RUN python3 - << 'PEOF'
import re, pathlib
ALIASES = [
    (r'\bnp\.int\b(?!\d|_|e)',      'int'),
    (r'\bnp\.float\b(?!\d|_|e|i)',  'float'),
    (r'\bnp\.bool\b(?!_|e)',        'bool'),
    (r'\bnp\.complex\b(?!\d|_)',    'complex'),
    (r'\bnp\.object\b(?!_)',        'object'),
    (r'\bnp\.str\b(?!_)',           'str'),
]
for f in pathlib.Path('/opt/SynthSeg').rglob('*.py'):
    src = f.read_text(errors='replace')
    new = src
    for pat, rep in ALIASES:
        new = re.sub(pat, rep, new)
    if new != src:
        f.write_text(new)
PEOF

WORKDIR /opt/SynthSeg
ENV CUDA_VISIBLE_DEVICES=-1
ENV TF_CPP_MIN_LOG_LEVEL=2
ENV PYTHONPATH=/opt/SynthSeg:/opt/SynthSeg/SynthSeg:/opt/SynthSeg/ext

CMD ["python3", "-c", "\
from SynthSeg.predict import predict; \
predict(\
  path_images='/data/T1.nii.gz', \
  path_segm='/data/seg.nii.gz', \
  path_model='/data/synthseg_2.0.h5', \
  path_volumes='/data/volumes.csv', \
  do_qc=False, cpu=True, threads=1)"]
EOF

# ── 2. Build (first time only, ~5 min) ───────────────────────────────────
docker build --platform linux/arm64 -t synthseg:2.0-arm64 ~/mri_data/

# ── 3. Run ────────────────────────────────────────────────────────────────
# Place T1.nii.gz and synthseg_2.0.h5 in ~/mri_data/ first
docker run --rm \
  --platform linux/arm64 \
  --memory="7g" --memory-swap="7g" \
  -v "$HOME/mri_data:/data" \
  synthseg:2.0-arm64
```

### Key flags

| Flag | Why |
|---|---|
| `--platform linux/arm64` | Native arm64 — no Rosetta overhead |
| `--memory=7g --memory-swap=7g` | Hard cap → Docker fails gracefully instead of kernel OOM-kill |
| `threads=1` | Single thread prevents multi-buffer RAM explosion |
| `do_qc=False` | Skips QC model, saves ~1 GB |
| `tensorflow-cpu` | CPU-only wheel, no CUDA libs |
| `CUDA_VISIBLE_DEVICES=-1` | Forces CPU even on GPU Macs |

### Expected runtime & memory (8 GB M-series Mac)

| Config | Peak RAM | Time (256³) |
|---|---|---|
| `threads=4, do_qc=True` | ~8–9 GB | 8–12 min — **OOM risk** |
| `threads=1, do_qc=True` | ~6–7 GB | 15–20 min — marginal |
| `threads=1, do_qc=False` | **~5–6 GB** | 12–18 min — **recommended** |

> If still OOM-killed: lower to `--memory=6g` and verify Docker Desktop has ≥ 8 GB in *Settings → Resources → Memory*.
